# Reading files and tables: paths, CSV, JSON, Excel and SQL

Python stage, lesson P9. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0.

**Goals:** explain the working directory and build a path with `Path`; read a CSV file into a table and name its encoding; read a JSON file into dictionaries and lists; read an Excel
sheet and the result of a SQL query into a table.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
The one cell that fails on purpose catches its error with `try` and `except`, so Run all never stops.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer from the cafe-sales data. Hints and solutions are folded away
under each exercise, free to open.

## Setup

In [ ]:
import math
import numbers

import numpy as np
import pandas as pd

# Run this cell as it is. It also holds the answer checker used by the exercise cells.
EXERCISES = ["Exercise 1", "Exercise 2", "Exercise 3"]


RESULTS = {}


def same(got, expected):
    """True when two values agree: floats within rounding, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k]) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps or {}).items():
        if same(got, wrong):
            print("Hint:", message)
    return False

## Data

In [ ]:
# The five cafe-sales tables (version 1), read from the release tag v0.1.0.
BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v0.1.0/datasets/cafe-sales/v1/"
stores = pd.read_csv(BASE + "stores.csv")
customers = pd.read_csv(BASE + "customers.csv")
products = pd.read_csv(BASE + "products.csv")
orders = pd.read_csv(BASE + "orders.csv")
order_items = pd.read_csv(BASE + "order_items.csv")

# Known answer from the dataset README: 3, 150, 16, 1,800 and 3,287 rows.
assert (len(stores), len(customers), len(products), len(orders), len(order_items)) == (3, 150, 16, 1800, 3287)
print("Data loaded: 3 stores, 150 customers, 16 products, 1,800 orders, 3,287 order lines.")

## Files for this lesson

In [ ]:
import json
import sqlite3
from pathlib import Path

# Run this cell as it is. It saves small files in a folder called cafe_data and builds a small database in memory,
# so that the units below have something to read. Your own files and databases already exist.
folder = Path("cafe_data")
folder.mkdir(exist_ok=True)
stores.to_csv(folder / "stores.csv", index=False)
customers.to_csv(folder / "customers.csv", index=False)
products.to_csv(folder / "products.csv", index=False)
(folder / "board_names.csv").write_text("product_id,board_name\n1,Café espresso\n3,Café au lait\n8,Chai latté\n", encoding="latin-1")
(folder / "till_names.csv").write_text("product_id,till_name\n7,Açaí green tea\n9,Piña colada iced tea\n", encoding="latin-1")
(folder / "order_1107.json").write_text("""{
  "order_id": 1107,
  "store": "Old Town",
  "ordered_at": "2026-01-15 16:57:00",
  "lines": [
    {"product": "Mocha", "quantity": 2, "discount_usd": 0.5},
    {"product": "Cappuccino", "quantity": 1, "discount_usd": 0.5}
  ]
}
""", encoding="utf-8")

connection = sqlite3.connect(":memory:")
for name, table in [("stores", stores), ("customers", customers), ("products", products), ("orders", orders), ("order_items", order_items)]:
    table.to_sql(name, connection, index=False)
print("Saved", len(list(folder.iterdir())), "files in", folder, "and built a database with 5 tables.")

## Unit 1: Where is the file?

**Predict first.** The setup cell saved the price list in a folder called `cafe_data`. What do these prints show: the path and whether it exists, whether a file with the same name exists next to the notebook, and whether each path is absolute?

In [ ]:
from pathlib import Path

price_file = Path("cafe_data") / "products.csv"
print(price_file, price_file.exists())
print(Path("products.csv").exists())
print(price_file.is_absolute(), Path.cwd().is_absolute())

## Unit 2: A CSV file, and how its letters were saved

**Predict first.** How many rows and columns does the price list have? What do its first three rows show?

In [ ]:
price_table = pd.read_csv(Path("cafe_data") / "products.csv")
print(price_table.shape)
price_table.head(3)

**Predict first.** The board names file was saved by the till in the encoding `latin-1`. What happens when `read_csv` reads it as it is, and what do the names look like when you name the encoding?

In [ ]:
board_file = Path("cafe_data") / "board_names.csv"
try:
    pd.read_csv(board_file)
except UnicodeDecodeError as error:
    print(type(error).__name__, "-", error.reason)
board_names = pd.read_csv(board_file, encoding="latin-1")
board_names["board_name"].tolist()

## Unit 3: A JSON file

**Predict first.** What do the two prints show, and what does the table of order lines look like?

In [ ]:
import json

order = json.loads((Path("cafe_data") / "order_1107.json").read_text(encoding="utf-8"))
print(order["store"], len(order["lines"]))
pd.DataFrame(order["lines"])

## Unit 4: An Excel sheet and a SQL query

**Predict first.** What shape does the `orders` sheet have? Which table does the query give back? The query is the `GROUP BY` of lesson S8.

In [ ]:
orders_excel = pd.read_excel(BASE + "cafe-sales.xlsx", sheet_name="orders")
print(orders_excel.shape)

query = """
SELECT store_id, COUNT(*) AS orders
FROM orders
GROUP BY store_id
ORDER BY store_id
"""
pd.read_sql(query, connection)

## Unit 5: Check: three routes, one table

**Predict first.** The same orders came in as a CSV file, as an Excel sheet and from a database. What are the three shapes?

In [ ]:
orders_csv = pd.read_csv(BASE + "orders.csv")
orders_sql = pd.read_sql("SELECT * FROM orders", connection)
print(orders_csv.shape, orders_excel.shape, orders_sql.shape)

## Finish it

### Exercise 1: Finish it

**Task:** How many customers does the file `cafe_data/customers.csv` hold? Its path is written for you. Replace the first `None` with the table that `read_csv` returns, then set `answer_1` to the number of rows of that table.

**Expected:** one whole number.

In [ ]:
customer_file = Path("cafe_data") / "customers.csv"
customers_table = None  # replace None: read customer_file with read_csv
answer_1 = None  # then set answer_1 to the number of rows of customers_table

In [ ]:
check("Exercise 1", "answer_1", 150, traps={5: "5 is the number of columns. shape is (rows, columns): which of the two do you need?"})

<details>
<summary>Hint</summary>

Unit 2 reads a CSV file by its path. `shape` is a pair: the rows first, then the columns.

</details>

<details>
<summary>Solution</summary>

```python
customers_table = pd.read_csv(customer_file)
answer_1 = customers_table.shape[0]
```

The answer is 150: one row for each loyalty-program customer. `shape[1]` is 5, the number of columns.

</details>

## On your own

### Exercise 2: On your own

**Task:** The workbook `cafe-sales.xlsx` holds all five tables, one sheet each. How many rows does its `order_items` sheet hold? Before you start, say in one sentence what kind of number the answer is. Read the sheet from `BASE + "cafe-sales.xlsx"` and set `answer_2`.

**Expected:** one whole number.

In [ ]:
# Exercise 2: read the order_items sheet of the workbook BASE + "cafe-sales.xlsx", then set answer_2 to its number of rows.

In [ ]:
check("Exercise 2", "answer_2", 3287, traps={3: "3 rows is another sheet: the stores sheet. Which sheet did read_excel read? Name the one you want."})

<details>
<summary>Hint</summary>

Unit 4 reads one sheet of this workbook. A workbook has several sheets, so say which one you want.

</details>

<details>
<summary>Solution</summary>

```python
order_lines = pd.read_excel(BASE + "cafe-sales.xlsx", sheet_name="order_items")
answer_2 = order_lines.shape[0]
```

The answer is 3287: one row for each product line of an order. Without `sheet_name`, the call reads another sheet and gives 3 rows.

</details>

## Check yourself

### Exercise 3: Check yourself: spot the flaw

**Task:** The till saved `cafe_data/till_names.csv` in the encoding `latin-1`. A colleague saw an error, added `encoding_errors="ignore"` and got a table without an error. Read the names again so that every letter is right, then set `answer_3` to the list of the two names.

**Expected:** a list of two names, both with their accents.

In [ ]:
till_file = Path("cafe_data") / "till_names.csv"
till_names = pd.read_csv(till_file, encoding_errors="ignore")  # the colleague's fix
answer_3 = None  # fix the line above, then set answer_3 to the list of the names in the till_name column

In [ ]:
check("Exercise 3", "answer_3", ["Açaí green tea", "Piña colada iced tea"], traps={("Aa green tea", "Pia colada iced tea"): "Ignoring the error threw the accented letters away. Name the encoding the file was saved in instead."})

<details>
<summary>Hint</summary>

Look at the names the colleague's table holds. Ignoring an error does not make the bytes right. The statement says how the file was saved.

</details>

<details>
<summary>Solution</summary>

```python
till_names = pd.read_csv(till_file, encoding="latin-1")
answer_3 = till_names["till_name"].tolist()
```

The answer is `['Açaí green tea', 'Piña colada iced tea']`. `encoding_errors="ignore"` dropped every byte it could not read as UTF-8 and left `'Aa green tea'`.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")